<a href="https://colab.research.google.com/github/DABMASTER-Brought-me-into-this/ForFunModel/blob/main/MyCustomModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Neural Network
When hypothesizing how Transformers are able to take a variable # of inputs (from 0 - context length) I predicted that they did so by considering every single combination of the tokens prior to it and did inference on that. While this model is completely unrealistic to be used with subword emebeddings, I am very curious on how it will fair with char level embeddings. I am also curious if I change the language to something other language with fewer characters (ex: Rotokas, or Toki Pona) as they have less characters.

In [ ]:
# Dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-07-02 00:09:50--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.24’

input.txt.24        100%[===================>]   1.06M  --.-KB/s    in 0.008s  

2026-07-02 00:09:50 (126 MB/s) - ‘input.txt.24’ saved [1115394/1115394]



In [ ]:
# Standrad Imports
import re
import math
import torch # Using Torch J for This One
import random
import itertools
import numpy as np
import torch.nn.functional as F

In [ ]:
# Hyper Parameter
context_length = 8
N_EMBD_MSK = 12
N_EMBD_CNN = 12
NDUM_TOK = 0.4
NUMBER_OF_NEURONS_L1_MSK = 128
NUMBER_OF_NEURONS_L2_MSK = 32
NUMBER_OF_NEURONS_L3_MSK = 8
NUMBER_OF_NEURONS_L1_CNN = 16
NUMBER_OF_NEURONS_L2_CNN = 8
BATCH_SIZE = 32
EPOCHS_MSK = 14

Note to Self: Figure out the science behind choosing Hyper parameters, because right now I am just picking numbers out of the blue, or numbers that I have seen used in similar networks I seen before.

In [ ]:
# Opening Dataset + Dataset Cleansing
with open('input.txt', 'r') as file:
  text = file.read().lower()
  text = re.sub('[^a-z ]', '', text)

In [ ]:
# Creating Mappings of tokens to string and vice versa
stoi = {letter: i for i, letter in enumerate(sorted(list(set(text))))}
itos = dict(zip(stoi.values(), stoi.keys()))

In [ ]:
# Encoding & Decoding Functions
encode = lambda text: list(map(lambda input: stoi[input], list(text)))
decode = lambda tokens: ''.join(list(map(lambda tok: itos[tok], tokens)))

In [ ]:
# Tokenizing the Dataset
tokenized_text = encode(text)
tokens = torch.tensor(tokenized_text + [0 for i in range(int((math.ceil(len(tokenized_text) / 8) - len(tokenized_text) / 8) * 8))])
targets_msk = torch.zeros_like(tokens)
tokens.shape

torch.Size([1020976])

In [ ]:
# Creating Dumbed Dataset
dumtok_ix = torch.randint(0, tokens.shape[0], size = (int(tokens.shape[0] * NDUM_TOK), )) # Indexes To Replace w/ Dum Tok
targets_msk[dumtok_ix] = 1
tokens[dumtok_ix] = torch.randint(0, len(itos), size = (int(tokens.shape[0] * NDUM_TOK), ))

In [ ]:
# Reshaping Tokens Into Batch, Context_length (AKA Time)
tokens = tokens.view(-1, context_length)
targets_msk = targets_msk.view(-1, context_length)
tokens.shape

torch.Size([127622, 8])

In [ ]:
from numpy.random.mtrand import noncentral_chisquare
from functools import lru_cache
### Creating PyTorch-Ish Classes
class Embedding:
  def __init__(self, vocab_size, n_embd):
    self.C = torch.ones(vocab_size, n_embd, requires_grad=True)

  def __call__(self, X):
    self.X = X
    self.out = self.C[self.X]
    return self.out

  def _parameters(self):
    return [self.C]

# ====================================================================================================

class Reshape:
  def __init__(self, shape):
    self.shape = shape

  def __call__(self, X):
    self.X = X
    self.out = self.X.view(self.shape)
    return self.out

  def _parameters(self):
    return []

# ====================================================================================================

class Conv1D:
  def __call__(self, X):
    self.X = X

    if self.X.ndim > 2:
      self.out = self.X.view(-1, self.X.shape[1]//2, self.X.shape[2] * 2)
      if self.X.shape[1] == 1:
        self.out = torch.squeeze(self.out, dim = 1)
    else:
      assert True, f"ERROR: X has {self.X.ndim} dimensions"

    return self.out

  def _parameters(self):
    return []

# ====================================================================================================

class Linear:
  def __init__(self, fan_in, fan_out, bias=False):
    self.W = torch.ones(fan_in, fan_out, requires_grad=True) # * 0.01 # Leaf Tensors are dum
    self.B = torch.zeros(fan_out, requires_grad=True) if bias else None

  def __call__(self, X):
    self.X = X
    self.out = self.X @ self.W
    if self.B is not None:
      self.out += self.B
    return self.out

  def _parameters(self):
    return [self.W, self.B] if self.B is not None else [self.W]

# ====================================================================================================

class BatchNorm:
  def __init__(self, fan_out, training = True):
    self.g = torch.ones(fan_out, requires_grad=True)
    self.b = torch.zeros(fan_out, requires_grad=True)
    self.running_mean = None
    self.running_var = None
    self.training = training

  def __call__(self, X):
    self.X = X

    if self.training:
      self.xmean = self.X.mean(axis = 0, keepdim=True)
      self.xvar = self.X.var(axis = 0, keepdim=True)

      if self.running_mean is None or self.running_var is None:
        self.running_mean = self.xmean
        self.running_var = self.xvar
      else:
        self.running_mean = self.running_mean * 0.9 + self.xmean * 0.1
        self.running_var = self.running_var * 0.9 + self.xvar * 0.1
    else:
       self.xmean = self.running_mean
       self.xvar = self.running_var

    self.prebn = (self.X - self.xmean)/torch.sqrt(self.xvar + 1e-5)
    self.out = self.g * self.prebn + self.b
    return self.out

  def _parameters(self):
    return [self.g, self.b]

# ====================================================================================================

class ReLu:
  def __call__(self, X):
    self.X = X
    self.out = torch.maximum(torch.zeros_like(self.X), self.X)
    return self.out

  def _parameters(self):
    return []

# ====================================================================================================

class SoftMax:
  def __call__(self, X):
    self.X = X
    x_max = torch.max(self.X, dim=1, keepdim=True).values
    logits_exp = torch.exp(self.X - x_max)
    self.out = logits_exp / torch.sum(logits_exp, dim=1, keepdim=True)
    return self.out

  def _parameters(self):
    return []

# ====================================================================================================

class Sigmoid:
  def __call__(self, X):
    self.X = X
    self.out = 1 / (1 + torch.e ** -(self.X))
    return self.out

  def _parameters(self):
    return []

# ====================================================================================================

class Model:
  def __init__(self, layers, lr):
    self.layers = layers
    self.parameters = [parameter for layer in self.layers for parameter in layer._parameters()]
    self.lr = lr

  def forward(self, X):
    self.out = X
    for layer in self.layers:
      self.out = layer(self.out)
    return self.out

  def update_parameters(self):
    for parameter in self.parameters:
      if parameter.grad is not None:
        parameter.data -= parameter.grad * self.lr

  def zero_grad(self):
    for parameter in self.parameters:
      if parameter.is_leaf:
        pass
      else:
        print(f"This parameter with shape {parameter.shape} is not a leaf tensor")
      if parameter.grad is not None:
        parameter.grad.zero_()

### Upon Mulling It Over
I mulled it over some, and I decided that this was dumb, because if you think about what I am actually doing theoretically, Im just feeding a lot of garbage data in a 1D CNN. So, instead, I think I will pivot this model by doing two things. The first is to make this model have multiple parts, kind of like a fusion network, but slightly different. I will build a Multi-Layer Perceptron that will take the inputs and create a mask of weights. This will tell the CNN before it ever gets anything inside of it, how much it should care about a token. This is in part developed by my prediction on how Karpathy will create his affinities for tokens with each other.

### Future Additions I am unsure about adding as of right now
So, every token outputed has a certain level of 'confidence' to it. For example, if you are using a character level model, and you observe the probs before you use multinomial to select a character, you can tell to a degree how confident your model is. If your model's probs are all ~ 1/27 (assuming your vocab size is 27) then your model is very very unconfident, but if there is like a single value that is 0.999999 and the rest is 0.000001 then your model is very confident. I dont know how I will make my model not output left to right YET, but I have an idea on how I want to measure this 'confidence.' The best way I can think of is standard deviation or a chi-squared test statistic (inspired by Batch Norm's z score) on each. I think chi-squared might be a tid bit better bcuz it can exaggerate the differences a little better than standard deviation or variance, but who knows (this prob alr exists tbh if it works how I think it will)

In [ ]:
# Creating Masking MLP First
mask_layers = [Embedding(len(stoi), N_EMBD_MSK), Reshape((BATCH_SIZE, -1)),
               Linear(context_length * N_EMBD_MSK, NUMBER_OF_NEURONS_L1_MSK), BatchNorm(NUMBER_OF_NEURONS_L1_MSK), ReLu(),
               Linear(NUMBER_OF_NEURONS_L1_MSK, NUMBER_OF_NEURONS_L2_MSK), BatchNorm(NUMBER_OF_NEURONS_L2_MSK), ReLu(),
               Linear(NUMBER_OF_NEURONS_L2_MSK, NUMBER_OF_NEURONS_L3_MSK, bias=True)]
mask_model = Model(mask_layers, 1e-3)

In [ ]:
# Training the Mask MLP First
# mask_model.mode(True)
for epoch in range(EPOCHS_MSK):
  for step in range(0, tokens.shape[0], BATCH_SIZE):
    # Mini Batching
    ix = torch.randint(0, tokens.shape[0], (BATCH_SIZE, ))
    xbatch = tokens[ix]
    ybatch = targets_msk[ix]

    # Forward Pass
    logits = mask_model.forward(xbatch)
    loss = F.binary_cross_entropy_with_logits(logits.float(), ybatch.float())

    # Backward Pass
    loss.backward()

    # Update Parameters
    mask_model.update_parameters()

    # Zero Out Grads
    mask_model.zero_grad()

  print(f"Epoch {epoch}: Loss {loss.item()}")

Epoch 0: Loss 0.7850188612937927
Epoch 1: Loss 0.665964663028717
Epoch 2: Loss 0.6696075797080994
Epoch 3: Loss 0.6476613283157349
Epoch 4: Loss 0.6265825629234314
Epoch 5: Loss 0.6238096356391907
Epoch 6: Loss 0.6408891677856445
Epoch 7: Loss 0.6529504060745239
